# **Check GPU Availability**


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
GPU available: []


# **Dependancies** **Installation**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

**Folder** **Creation**

In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)

print("Folder created successfully!")

Folder created successfully!


In [ ]:
!wget -O data/raw/human_activity.zip "https://archive.ics.uci.edu/static/public/341/smartphone+based+recognition+of+human+activities+and+postural+transitions.zip"


--2026-09-18 17:21:26--  https://archive.ics.uci.edu/static/public/341/smartphone+based+recognition+of+human+activities+and+postural+transitions.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘data/raw/human_activity.zip’

data/raw/human_acti     [          <=>       ]  75.91M  40.3MB/s    in 1.9s    

2026-09-18 17:21:28 (40.3 MB/s) - ‘data/raw/human_activity.zip’ saved [79596192]



In [12]:
!unzip -q data/raw/human_activity.zip -d data/raw/

replace data/raw/features_info.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/raw/README.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [13]:
!ls -R data/raw/


data/raw/:
activity_labels.txt  features.txt	 RawData     Test
features_info.txt    human_activity.zip  README.txt  Train

data/raw/RawData:
acc_exp01_user01.txt  acc_exp42_user21.txt   gyro_exp22_user11.txt
acc_exp02_user01.txt  acc_exp43_user21.txt   gyro_exp23_user11.txt
acc_exp03_user02.txt  acc_exp44_user22.txt   gyro_exp24_user12.txt
acc_exp04_user02.txt  acc_exp45_user22.txt   gyro_exp25_user12.txt
acc_exp05_user03.txt  acc_exp46_user23.txt   gyro_exp26_user13.txt
acc_exp06_user03.txt  acc_exp47_user23.txt   gyro_exp27_user13.txt
acc_exp07_user04.txt  acc_exp48_user24.txt   gyro_exp28_user14.txt
acc_exp08_user04.txt  acc_exp49_user24.txt   gyro_exp29_user14.txt
acc_exp09_user05.txt  acc_exp50_user25.txt   gyro_exp30_user15.txt
acc_exp10_user05.txt  acc_exp51_user25.txt   gyro_exp31_user15.txt
acc_exp11_user06.txt  acc_exp52_user26.txt   gyro_exp32_user16.txt
acc_exp12_user06.txt  acc_exp53_user26.txt   gyro_exp33_user16.txt
acc_exp13_user07.txt  acc_exp54_user27.txt   gyro_exp34

In [14]:
import pandas as pd

activity_labels = pd.read_csv(
    "data/raw/activity_labels.txt",
    sep=r"\s+",
    header=None,
    names=["label", "activity"]
)

print(activity_labels)

    label            activity
0       1             WALKING
1       2    WALKING_UPSTAIRS
2       3  WALKING_DOWNSTAIRS
3       4             SITTING
4       5            STANDING
5       6              LAYING
6       7        STAND_TO_SIT
7       8        SIT_TO_STAND
8       9          SIT_TO_LIE
9      10          LIE_TO_SIT
10     11        STAND_TO_LIE
11     12        LIE_TO_STAND


**Load the Training** **Data**

In [15]:
X_train = pd.read_csv(
    "data/raw/Train/X_train.txt",
    sep=r"\s+",
    header=None
)

y_train = pd.read_csv(
    "data/raw/Train/y_train.txt",
    sep=r"\s+",
    header=None,
    names=["label"]
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (7767, 561)
y_train shape: (7767, 1)


561-dimensional feature vector.

In [19]:
print("\nFirst 5 rows of X_train:")
display(X_train.head(10))

print("\nFirst 10 labels:")
display(y_train.head(50))


First 5 rows of X_train:


,0,1,2,3,4,5,6,7,8,9,...,551,552,553,554,555,556,557,558,559,560
0,0.043580,-0.005970,-0.035054,-0.995381,-0.988366,-0.937382,-0.995007,-0.988816,-0.953325,-0.794796,...,-0.012236,-0.314848,-0.713308,-0.112754,0.030400,-0.464761,-0.018446,-0.841559,0.179913,-0.051718
1,0.039480,-0.002131,-0.029067,-0.998348,-0.982945,-0.971273,-0.998702,-0.983315,-0.974000,-0.802537,...,0.202804,-0.603199,-0.860677,0.053477,-0.007435,-0.732626,0.703511,-0.845092,0.180261,-0.047436
2,0.039978,-0.005153,-0.022651,-0.995482,-0.977314,-0.984760,-0.996415,-0.975835,-0.985973,-0.798477,...,0.440079,-0.404427,-0.761847,-0.118559,0.177899,0.100699,0.808529,-0.849230,0.180610,-0.042271
3,0.039785,-0.011809,-0.028916,-0.996194,-0.988569,-0.993256,-0.996994,-0.988526,-0.993135,-0.798477,...,0.430891,-0.138373,-0.491604,-0.036788,-0.012892,0.640011,-0.485366,-0.848947,0.181907,-0.040826
4,0.038758,-0.002289,-0.023863,-0.998241,-0.986774,-0.993115,-0.998216,-0.986479,-0.993825,-0.801982,...,0.137735,-0.366214,-0.702490,0.123320,0.122542,0.693578,-0.615971,-0.848164,0.185124,-0.037080
5,0.038988,0.004109,-0.017340,-0.997438,-0.993485,-0.996692,-0.997522,-0.993494,-0.996916,-0.801982,...,0.074999,-0.554902,-0.844224,0.082632,-0.143439,0.275041,-0.368224,-0.849927,0.184795,-0.035326
6,0.039897,-0.005324,-0.020457,-0.997024,-0.977313,-0.987782,-0.996898,-0.977450,-0.989391,-0.800606,...,0.191486,-0.235576,-0.571126,-0.212754,-0.230622,0.014637,-0.189512,-0.852441,0.182142,-0.036203
7,0.039082,-0.016047,-0.030241,-0.996662,-0.976996,-0.986672,-0.996380,-0.977594,-0.989310,-0.800606,...,0.182731,-0.104337,-0.432022,-0.020888,0.593996,-0.561871,0.467383,-0.851309,0.183751,-0.035176
8,0.039026,-0.007410,-0.027301,-0.997431,-0.973190,-0.988183,-0.997491,-0.971557,-0.990156,-0.800245,...,0.347118,-0.286366,-0.579474,0.012954,0.080936,-0.234313,0.117797,-0.848270,0.188955,-0.030594
9,0.040354,0.004245,-0.017932,-0.994906,-0.981181,-0.990046,-0.995300,-0.982483,-0.990920,-0.799717,...,0.303948,0.306076,0.115919,-0.020590,-0.127730,-0.482871,-0.070670,-0.848592,0.190283,-0.027667



First 10 labels:


,label
0,5
1,5
2,5
3,5
4,5
5,5
6,5
7,5
8,5
9,5


In [20]:
features = pd.read_csv(
    "data/raw/features.txt",
    sep=r"\s+",
    header=None,
    names=["index", "feature"]
)

print("Number of features:", len(features))
display(features.head(20))

Number of features: 561


,index,feature
0,tBodyAcc-Mean-1,NaN
1,tBodyAcc-Mean-2,NaN
2,tBodyAcc-Mean-3,NaN
3,tBodyAcc-STD-1,NaN
4,tBodyAcc-STD-2,NaN
5,tBodyAcc-STD-3,NaN
6,tBodyAcc-Mad-1,NaN
7,tBodyAcc-Mad-2,NaN
8,tBodyAcc-Mad-3,NaN
9,tBodyAcc-Max-1,NaN
